In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%%capture
!pip install pyarrow

## **01. Read data**

In [ ]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import torch

TEACHER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
TEACHER_RUN_NAME = "bge-reranker-v2-m3"

MAX_LENGTH = 2048

BATCH_SIZE = 32

RERANK_DEPTH = 50

RERANK_FULL_CORPUS = False

EVALUATE_TEST_SPLIT = False   

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

BM25_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BM25/output"
OUTPUT_DIR = RETRIEVER_DIR / "03_Finetuning/teacher"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(RETRIEVER_DIR))

import retriever_common as rc

print("TOP_K_LIST:", rc.TOP_K_LIST)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))

TOP_K_LIST: [1, 3, 5, 10, 15, 20, 25, 30]
Device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [4]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

bm25_ranking = {}

for split_name in ['val', 'test']:
  path = BM25_OUTPUT_DIR / f'{split_name}_ranking.parquet'

  if path.exists():
    bm25_ranking[split_name] = pd.read_parquet(path)
    print(f"{split_name}_ranking: {len(bm25_ranking[split_name]):,} dòng")
  else:
    print(f"Không tìm thấy {path} — chạy BM25.ipynb trước.")

doc_id_to_text = dict(zip(corpus['doc_id'], corpus['embedding_text']))
query_id_to_text = dict(zip(query['query_id'], query['query']))

assert 'val' in bm25_ranking, "Bắt buộc phải có val_ranking.parquet của BM25."

val_ranking: 46,650 dòng
test_ranking: 47,900 dòng


In [ ]:
bm25_summary = {}

for split_name, ranking in bm25_ranking.items():
  per_query = rc.evaluation(ranking, rc.build_qrels_map(qrels[qrels['split'] == split_name]))
  bm25_summary[split_name] = rc.summarize(per_query)

  print(f"[BM25 {split_name}] " + "  ".join(
      f"hit@{k}={bm25_summary[split_name][f'hit@{k}']:.4f}" for k in [1, 5, 10, 30]
  ))

[BM25 val] hit@1=0.7128  hit@5=0.9089  hit@10=0.9464  hit@30=0.9753
[BM25 test] hit@1=0.6691  hit@5=0.8904  hit@10=0.9290  hit@30=0.9656


## **02. Model**

In [6]:
from sentence_transformers import CrossEncoder

teacher = CrossEncoder(
    TEACHER_MODEL_NAME,
    max_length=MAX_LENGTH,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    model_kwargs={'torch_dtype': 'float16'} if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

print("Teacher:", TEACHER_MODEL_NAME)
print("max_length:", teacher.max_length)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Teacher: BAAI/bge-reranker-v2-m3
max_length: 2048


/tmp/ipykernel_2816/1092616902.py:12: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  print("max_length:", teacher.max_length)


In [ ]:
smoke_query_id = query.loc[query['split'] == 'val', 'query_id'].iloc[0]
smoke_query_text = query_id_to_text[smoke_query_id]

smoke_positive_doc_id = sorted(rc.build_qrels_map(qrels[qrels['split'] == 'val'])[smoke_query_id])[0]

# Lấy một passage rõ ràng không liên quan: passage BM25 xếp cuối top-50.
smoke_ranking = bm25_ranking['val']
smoke_negative_doc_id = smoke_ranking.loc[
    smoke_ranking['query_id'] == smoke_query_id
].sort_values('rank')['doc_id'].iloc[-1]

smoke_scores = teacher.predict([
    (smoke_query_text, doc_id_to_text[smoke_positive_doc_id]),
    (smoke_query_text, doc_id_to_text[smoke_negative_doc_id]),
])

print("Câu hỏi :", smoke_query_text)
print("Điểm positive:", round(float(smoke_scores[0]), 4))
print("Điểm negative:", round(float(smoke_scores[1]), 4))

assert smoke_scores[0] > smoke_scores[1], (
    "Teacher chấm positive thấp hơn negative — kiểm tra lại thứ tự (query, passage)."
)

assert abs(float(smoke_scores[0]) - float(smoke_scores[1])) > 1e-4, (
    "Teacher trả về điểm gần như bằng nhau — có thể model chưa nạp đúng."
)

longest_passage = max(doc_id_to_text.values(), key=len)
longest_token_count = len(teacher.tokenizer.encode(longest_passage, add_special_tokens=True))

print(f"\nPassage dài nhất: {longest_token_count:,} token / max_length = {teacher.max_length:,}")

assert longest_token_count <= teacher.max_length, (
    f"Passage dài nhất ({longest_token_count}) vượt max_length ({teacher.max_length}) — tăng MAX_LENGTH."
)

print("\nSmoke test đã pass.")

Câu hỏi : Sinh viên chưa hết thời gian tối đa hoàn thành khóa học quy định tại Điều 6 của Quy chế này, đã hoàn thành các học phần trong chương trình đào tạo có nguyện vọng xin thôi học theo diện này thì phải làm gì?
Điểm positive: 0.9995
Điểm negative: 0.0371

Passage dài nhất: 1,557 token / max_length = 2,048

Smoke test đã pass.


/tmp/ipykernel_2816/398170930.py:35: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  print(f"\nPassage dài nhất: {longest_token_count:,} token / max_length = {teacher.max_length:,}")
/tmp/ipykernel_2816/398170930.py:37: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  assert longest_token_count <= teacher.max_length, (


## **03. Rerank trên val**

In [ ]:
def score_pairs(pair_list: list[tuple[str, str]], batch_size: int = BATCH_SIZE) -> np.ndarray:
  return np.asarray(
      teacher.predict(pair_list, batch_size=batch_size, show_progress_bar=True), dtype=np.float32
  )


def rerank_ranking(ranking_dataframe: pd.DataFrame, depth: int) -> pd.DataFrame:
  """Chấm lại `depth` ứng viên đầu của mỗi câu hỏi rồi xếp hạng lại."""
  candidates = ranking_dataframe[ranking_dataframe['rank'] <= depth].copy()

  pair_list = [
      (query_id_to_text[query_id], doc_id_to_text[doc_id])
      for query_id, doc_id in zip(candidates['query_id'], candidates['doc_id'])
  ]

  candidates['teacher_score'] = score_pairs(pair_list)

  candidates = candidates.sort_values(
      ['query_id', 'teacher_score'], ascending=[True, False], kind='mergesort'
  )

  candidates['rank'] = candidates.groupby('query_id').cumcount() + 1
  candidates['score'] = candidates['teacher_score']

  return candidates[['query_id', 'doc_id', 'rank', 'score', 'split']].reset_index(drop=True)

In [9]:
validation_qrels_map = rc.build_qrels_map(qrels[qrels['split'] == 'val'])

start_time = time.perf_counter()

validation_reranked = rerank_ranking(bm25_ranking['val'], depth=RERANK_DEPTH)

rerank_seconds = time.perf_counter() - start_time

validation_per_query = rc.evaluation(validation_reranked, validation_qrels_map)

rc.assert_metrics_are_sane(validation_per_query)

teacher_summary = rc.summarize(validation_per_query)

number_of_pairs_scored = int((bm25_ranking['val']['rank'] <= RERANK_DEPTH).sum())

print(f"\nĐã chấm {number_of_pairs_scored:,} cặp trong {rerank_seconds / 60:.1f} phút "
      f"({number_of_pairs_scored / rerank_seconds:.0f} cặp/giây)")

print("\n" + "  ".join(f"hit@{k}={teacher_summary[f'hit@{k}']:.4f}" for k in [1, 5, 10, 30]))

Batches:   0%|          | 0/1458 [00:00<?, ?it/s]


Đã chấm 46,650 cặp trong 2.0 phút (391 cặp/giây)

hit@1=0.8510  hit@5=0.9732  hit@10=0.9818  hit@30=0.9850


## **04. Compate to BM25**

In [10]:
comparison_records = []

for k in [1, 3, 5, 10, 15, 20, 25, 30]:
  comparison_records.append({
      'metric': f'hit@{k}',
      'bm25': round(bm25_summary['val'][f'hit@{k}'], 4),
      'teacher_rerank': round(teacher_summary[f'hit@{k}'], 4),
      'delta': round(teacher_summary[f'hit@{k}'] - bm25_summary['val'][f'hit@{k}'], 4),
  })

for name in ['mrr@10', 'ndcg@10']:
  comparison_records.append({
      'metric': name,
      'bm25': round(bm25_summary['val'][name], 4),
      'teacher_rerank': round(teacher_summary[name], 4),
      'delta': round(teacher_summary[name] - bm25_summary['val'][name], 4),
  })

display(pd.DataFrame(comparison_records))

,metric,bm25,teacher_rerank,delta
0,hit@1,0.7128,0.8510,0.1383
1,hit@3,0.8703,0.9571,0.0868
2,hit@5,0.9089,0.9732,0.0643
3,hit@10,0.9464,0.9818,0.0354
4,hit@15,0.9561,0.9839,0.0279
5,hit@20,0.9657,0.9839,0.0182
6,hit@25,0.9668,0.9850,0.0182
7,hit@30,0.9753,0.9850,0.0096
8,mrr@10,0.7970,0.9050,0.1081
9,ndcg@10,0.8337,0.9243,0.0907


In [ ]:
bm25_per_query = rc.evaluation(bm25_ranking['val'], validation_qrels_map)

bootstrap_records = []

for metric in ['hit@1', 'hit@5', 'mrr@10']:
  result = rc.paired_bootstrap(validation_per_query, bm25_per_query, metric=metric)

  bootstrap_records.append({
      'metric': metric,
      'teacher': round(result['mean_a'], 4),
      'bm25': round(result['mean_b'], 4),
      'difference': round(result['difference'], 4),
      'ci_95': f"[{result['ci_lower_95']:+.4f}, {result['ci_upper_95']:+.4f}]",
      'p_value': round(result['p_value'], 4),
      'significant': result['significant_at_0.05'],
  })

display(pd.DataFrame(bootstrap_records))

,metric,teacher,bm25,difference,ci_95,p_value,significant
0,hit@1,0.8510,0.7128,0.1383,"[+0.1115, +0.1661]",0.0,True
1,hit@5,0.9732,0.9089,0.0643,"[+0.0482, +0.0815]",0.0,True
2,mrr@10,0.9050,0.7970,0.1081,"[+0.0898, +0.1266]",0.0,True


### 4.1. Đánh giá có nên dùng model này không?

In [12]:
teacher_hit_at_1 = float(teacher_summary['hit@1'])
bm25_hit_at_1 = float(bm25_summary['val']['hit@1'])

print(f"Teacher hit@1 : {teacher_hit_at_1:.4f}")
print(f"BM25    hit@1 : {bm25_hit_at_1:.4f}")
print(f"Chênh lệch    : {teacher_hit_at_1 - bm25_hit_at_1:+.4f}")
print()

if teacher_hit_at_1 < bm25_hit_at_1:
  verdict = 'REJECT'
  print("DỪNG LẠI. Teacher yếu hơn BM25 trên miền này.")
  print("Chưng cất nó sẽ cho kết quả tệ hơn cả cách của paper gốc.")
  print("Hướng xử lý: fine-tune teacher trên 7.697 cặp train trước, hoặc thử")
  print("Alibaba-NLP/gte-multilingual-reranker-base, hoặc giữ nguyên teacher = BM25 (L1).")
elif teacher_hit_at_1 < 0.80:
  verdict = 'FINETUNE_TEACHER'
  print("Teacher hơn BM25 nhưng chưa nhiều.")
  print("Nên fine-tune teacher trên tập train trước khi dùng làm nhãn (1-2 epoch, lr thấp).")
  print("Vẫn có thể tiến hành L2 với teacher zero-shot để có mốc so sánh.")
else:
  verdict = 'ACCEPT'
  print("Teacher đủ khoẻ. Tiến hành 04b_TeacherScoring.ipynb.")

print(f"\nverdict = {verdict}")

Teacher hit@1 : 0.8510
BM25    hit@1 : 0.7128
Chênh lệch    : +0.1383

Teacher đủ khoẻ. Tiến hành 04b_TeacherScoring.ipynb.

verdict = ACCEPT


In [13]:
depth_records = []

for depth in [5, 10, 20, 50]:
  depth_reranked = rerank_ranking(bm25_ranking['val'], depth=depth)

  depth_summary = rc.summarize(rc.evaluation(depth_reranked, validation_qrels_map))

  depth_records.append({
      'rerank_depth': depth,
      'pairs_scored': int((bm25_ranking['val']['rank'] <= depth).sum()),
      'hit@1': round(depth_summary['hit@1'], 4),
      'hit@5': round(depth_summary['hit@5'], 4),
      'mrr@10': round(depth_summary['mrr@10'], 4),
      'bm25_ceiling_hit': round(bm25_summary['val'][f'hit@{depth}'], 4) if f'hit@{depth}' in bm25_summary['val'] else None,
  })

display(pd.DataFrame(depth_records))

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Batches:   0%|          | 0/292 [00:00<?, ?it/s]

Batches:   0%|          | 0/584 [00:00<?, ?it/s]

Batches:   0%|          | 0/1458 [00:00<?, ?it/s]

,rerank_depth,pairs_scored,hit@1,hit@5,mrr@10,bm25_ceiling_hit
0,5,4665,0.8221,0.9089,0.8609,0.9089
1,10,9330,0.8371,0.9464,0.8851,0.9464
2,20,18660,0.8414,0.9603,0.8933,0.9657
3,50,46650,0.8510,0.9732,0.9050,NaN


### 4.3. Rerank toan bo 294 passage

In [14]:
if RERANK_FULL_CORPUS:
  validation_query = query[query['split'] == 'val']

  all_doc_ids = corpus['doc_id'].tolist()

  full_pair_list = [
      (query_id_to_text[query_id], doc_id_to_text[doc_id])
      for query_id in validation_query['query_id']
      for doc_id in all_doc_ids
  ]

  print(f"Chấm {len(full_pair_list):,} cặp...")

  full_scores = score_pairs(full_pair_list).reshape(len(validation_query), len(all_doc_ids))

  full_reranked = rc.rankings_from_score_matrix(
      full_scores, validation_query['query_id'].tolist(), np.asarray(all_doc_ids), 'val'
  )

  full_per_query = rc.evaluation(full_reranked, validation_qrels_map)

  full_summary = rc.summarize(full_per_query)

  print("\nRerank toàn corpus: " + "  ".join(
      f"hit@{k}={full_summary[f'hit@{k}']:.4f}" for k in [1, 5, 10]
  ))
  print(f"Rerank BM25 top-{RERANK_DEPTH}: hit@1={teacher_summary['hit@1']:.4f}")
  print("\nNếu hai con số hit@1 gần nhau, tầng BM25 không làm mất gì —")
  print("giữ kiến trúc hai tầng vì nó rẻ hơn nhiều lần.")
else:
  print("Bỏ qua. Bật RERANK_FULL_CORPUS = True ở mục 1 nếu muốn chạy.")

Bỏ qua. Bật RERANK_FULL_CORPUS = True ở mục 1 nếu muốn chạy.


## **05. Save**

In [17]:
latency_query_ids = query.loc[query['split'] == 'val', 'query_id'].tolist()[:50]

latency_records = []

for depth in [5, 10, 50]:
  # Chạy nóng
  warmup_pairs = [
      (query_id_to_text[latency_query_ids[0]], doc_id_to_text[doc_id])
      for doc_id in bm25_ranking['val'].loc[
          bm25_ranking['val']['query_id'] == latency_query_ids[0]
      ].sort_values('rank')['doc_id'].tolist()[:depth]
  ]
  teacher.predict(warmup_pairs, batch_size=BATCH_SIZE, show_progress_bar=False)

  start_time = time.perf_counter()

  for query_id in latency_query_ids:
    candidate_doc_ids = bm25_ranking['val'].loc[
        bm25_ranking['val']['query_id'] == query_id
    ].sort_values('rank')['doc_id'].tolist()[:depth]

    teacher.predict(
        [(query_id_to_text[query_id], doc_id_to_text[doc_id]) for doc_id in candidate_doc_ids],
        batch_size=BATCH_SIZE, show_progress_bar=False,
    )

  elapsed = time.perf_counter() - start_time

  latency_records.append({
      'rerank_depth': depth,
      'latency_ms_per_query': round(elapsed / len(latency_query_ids) * 1000, 1),
      'queries_per_second': round(len(latency_query_ids) / elapsed, 2),
  })

latency_df = pd.DataFrame(latency_records)

display(latency_df)

print("Tham chiếu: BM25 = 0.71 ms/query")

,rerank_depth,latency_ms_per_query,queries_per_second
0,5,43.4,23.02
1,10,81.3,12.29
2,50,327.6,3.05


Tham chiếu: BM25 = 0.71 ms/query


In [18]:
if EVALUATE_TEST_SPLIT and 'test' in bm25_ranking:
  test_reranked = rerank_ranking(bm25_ranking['test'], depth=RERANK_DEPTH)
  test_per_query = rc.evaluation(test_reranked, rc.build_qrels_map(qrels[qrels['split'] == 'test']))

  ranking_by_split = {'val': validation_reranked, 'test': test_reranked}
  per_query_by_split = {'val': validation_per_query, 'test': test_per_query}
else:
  ranking_by_split = {'val': validation_reranked}
  per_query_by_split = {'val': validation_per_query}

In [19]:
latency_records = []

run_name = f"BM25+{TEACHER_RUN_NAME}__rerank{RERANK_DEPTH}"

run_config = {
    'run_name': run_name,
    'stage': 'teacher_evaluation',
    'teacher_model': TEACHER_MODEL_NAME,
    'max_length': MAX_LENGTH,
    'rerank_depth': RERANK_DEPTH,
    'first_stage': 'BM25 (k1=0.9, b=0.75)',
    'verdict': verdict,
    'teacher_hit@1': teacher_hit_at_1,
    'bm25_hit@1': bm25_hit_at_1,
    'pairs_scored': number_of_pairs_scored,
    'rerank_minutes': round(rerank_seconds / 60, 2),
    'latency': latency_records,
    'corpus_size': len(corpus),
}

summary_dataframe = rc.save_run(
    OUTPUT_DIR, run_name, ranking_by_split, per_query_by_split, run_config
)

display(summary_dataframe[['run_name', 'split', 'hit@1', 'hit@5', 'hit@10', 'mrr@10', 'ndcg@10']].round(4))

print("\nverdict:", verdict)

,run_name,split,hit@1,hit@5,hit@10,mrr@10,ndcg@10
0,BM25+bge-reranker-v2-m3__rerank50,val,0.851,0.9732,0.9818,0.905,0.9243



verdict: ACCEPT
